In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import multiprocessing
import os
import pickle

try:
    cores = int(os.environ["SLURM_JOB_CPUS_PER_NODE"])
except:
    cores = multiprocessing.cpu_count()


os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(cores)

import jax
import jax.numpy as jnp
from astropy.table import Table
import numpy as np
import pandas as pd
import gpjax as gpx
from gpjax.kernels import RBF, Linear, Periodic, Polynomial
from jax import jit
from jaxoplanet import orbits
from jaxoplanet.light_curves import LimbDarkLightCurve
from jaxopt import ScipyMinimize
from tensorflow_probability.substrates.jax.distributions import Normal
from tqdm import tqdm

from gallifrey.kernels import OrnsteinUhlenbeck
from gallifrey.kernelsearch import KernelSearch, print_kernel_summary, get_trainables
from gallifrey.mcmc import nuts_warmup, run_mcmc
from gallifrey.inference import log_likelihood_function

In [ ]:
jax.config.update("jax_enable_x64", True)
rng_key = jax.random.PRNGKey(42)

## Load Data

In [ ]:
df = Table.read("../data/external/WASP_94Ab.fit").to_pandas()

t = df["Time"].to_numpy()
t_min = np.amin(t)
t -= t_min

# white lightcurve
white_lc = df["FluxWL"].to_numpy().T
white_lc_err = df["e_FluxWL"].to_numpy().T

# spectroscopic light curves
y = df.iloc[:, 1:-2:2].to_numpy().T
yerr = df.iloc[:, 2:-1:2].to_numpy().T

# mask out transit
mask = np.ones_like(t, dtype=bool)
mask[100:324] = False

# reference parameter and limb darkening parameter
reference = pd.read_csv("../data/external/WASP_94Ab_reference.csv")

## Get GP models for white light curve

In [ ]:
kernel_library = [
    Linear(),
    RBF(),
    OrnsteinUhlenbeck(),
    Periodic(),
]

In [ ]:
tree = KernelSearch(
    kernel_library,
    X=jnp.array(t[mask]),
    y=jnp.array(white_lc[mask]),
    obs_stddev=jnp.amax(white_lc_err[mask]),
    verbosity=1,
)

# model = tree.search(
#     depth=10,
#     n_leafs=3,
#     patience=1,
# )

In [ ]:
model_name = "wasp94Ab_gpmodel"
# with open(f"../data/processed/observational_data/gp_models/{model_name}", "wb") as file:
#     pickle.dump(model, file)
with open(f"../data/processed/observational_data/gp_models/{model_name}", "rb") as file:
    model = pickle.load(file)
print_kernel_summary(model)

In [ ]:
# recreate kernel using Polynomial kernel with same properties for computational efficiency

parameter = get_trainables(model)

new_kernel = OrnsteinUhlenbeck(lengthscale=parameter[0], variance=parameter[1]) + (
    Polynomial(degree=4, shift=0.0, variance=parameter[2]).replace_trainable(
        shift=False
    )
    * RBF(lengthscale=parameter[3]).replace_trainable(variance=False)
)

model = model.likelihood * gpx.gps.Prior(
    mean_function=model.prior.mean_function, kernel=new_kernel
)

## Fit white curve parameter

In [ ]:
def white_lc_model(t, params):
    central = orbits.keplerian.Central(
        mass=params[0],
        radius=params[1],
    )

    # The light curve calculation requires an orbit
    orbit = orbits.keplerian.Body(
        central=central,
        period=3.9501907,
        radius=params[4] * central.radius,
        inclination=jnp.deg2rad(params[2]),
        time_transit=params[3],
    )

    lc = LimbDarkLightCurve([params[5], 0.862]).light_curve(orbit, t=t)
    return lc


white_lc_log_likelihood = log_likelihood_function(
    model,
    white_lc_model,
    t,
    white_lc,
    mask,
    fix_gp=False,
    compile=True,
    negative=True,
)

white_lc_solve = ScipyMinimize(fun=white_lc_log_likelihood, method="l-bfgs-b").run(
    {
        "gp_parameter": get_trainables(model, unconstrain=True),
        "lc_parameter": jnp.array([1.45, 1.653, 89.3, 0.18, 0.11, 0.53]),
    }
)
white_lc_parameter = white_lc_solve.params["lc_parameter"]

## Define LC model

In [ ]:
def get_lc_model(u2):
    def lc_model(t, params):
        central = orbits.keplerian.Central(
            mass=white_lc_parameter[0],
            radius=white_lc_parameter[1],
        )

        # The light curve calculation requires an orbit
        orbit = orbits.keplerian.Body(
            central=central,
            period=3.9501907,
            radius=params[0] * central.radius,
            inclination=jnp.deg2rad(white_lc_parameter[2]),
            time_transit=white_lc_parameter[3],
        )

        lc = LimbDarkLightCurve([params[1], u2]).light_curve(orbit, t=t)
        return lc

    return lc_model

## Define likelihood, prior, posterior

In [ ]:
def get_logprob(gp_model, y, yerr, u1, u2, initial_position=None):
    if initial_position is None:
        initial_position = {
            "gp_parameter": get_trainables(gp_model, unconstrain=True),
            "lc_parameter": jnp.array([0.10, u1]),
        }

    param_priors = {
        "gp_parameter": Normal(
            loc=initial_position["gp_parameter"],
            scale=0.1 * jnp.abs(initial_position["gp_parameter"]),
        ),
        "lc_parameter": Normal(
            loc=initial_position["lc_parameter"],
            scale=[0.2, 0.05],
        ),
    }

    # define light curve model
    lc_model = jit(get_lc_model(u2))

    # update observational uncertainty data uncertainty
    gp_model = (
        gp_model.likelihood.replace(obs_stddev=jnp.amax(yerr[mask])) * model.prior
    )

    log_likelihood = log_likelihood_function(
        gp_model,
        lc_model,
        t,
        y,
        mask,
        fix_gp=False,
        compile=True,
    )

    @jit
    def log_priors(params):
        gp_log_priors = param_priors["gp_parameter"].log_prob(params["gp_parameter"])
        lc_log_priors = param_priors["lc_parameter"].log_prob(params["lc_parameter"])
        return jnp.sum(gp_log_priors) + jnp.sum(lc_log_priors)

    @jit
    def log_probability(params):
        return log_likelihood(params) + log_priors(params)

    return log_probability, initial_position

## Fits

In [ ]:
parameter_solutions = []
for i in tqdm(range(len(y))):
    log_probability, initial_position = get_logprob(
        model,
        y[i],
        yerr[i],
        reference["u1_linear"].iloc[i],
        reference["u2"].iloc[i],
    )

    solve = ScipyMinimize(
        fun=jit(lambda par: -log_probability(par)), method="l-bfgs-b"
    ).run(initial_position)
    parameter_solutions.append(solve.params)

In [ ]:
# import matplotlib.pyplot as plt

# plt.scatter(range(len(y)), [sol["lc_parameter"][0] for sol in parameter_solutions])
# plt.scatter(range(len(y)), reference["Rp_linear"])
# plt.scatter(range(len(y)), reference["Rp_gp"])

# plt.figure()
# plt.scatter(
#     range(len(y)),
#     (reference["Rp_linear"] - [sol["lc_parameter"][0] for sol in parameter_solutions])
#     / reference["Rp_linear"],
# )
# plt.scatter(
#     range(len(y)),
#     (reference["Rp_gp"] - [sol["lc_parameter"][0] for sol in parameter_solutions])
#     / reference["Rp_gp"],
# )

## Run MCMC

In [ ]:
# Adapted from BlackJax's introduction notebook.
num_adapt = 300
num_samples = 80
num_chains = cores

In [ ]:
rng_key, warmup_key = jax.random.split(rng_key, 2)

# run nuts adaption on white lc
log_probability, initial_position = get_logprob(
    model,
    white_lc,
    white_lc_err,
    0.5326,
    0.0862,
)

state, parameters = nuts_warmup(
    warmup_key,
    white_lc_log_likelihood,
    initial_position,
    num_steps=num_adapt,
)

In [ ]:
chains = {"gp_parameter": [], "lc_parameter": []}

for i in tqdm(range(len(y))):
    log_probability, initial_position = get_logprob(
        model,
        y[i],
        yerr[i],
        reference["u1_linear"].iloc[i],
        reference["u2"].iloc[i],
        initial_position=parameter_solutions[i],
    )

    rng_key, sample_key = jax.random.split(rng_key, 2)

    initial_positions = {
        "gp_parameter": jnp.tile(initial_position["gp_parameter"], (num_chains, 1)),
        "lc_parameter": jnp.tile(initial_position["lc_parameter"], (num_chains, 1)),
    }

    final_state, state_history, info_history = run_mcmc(
        sample_key,
        log_probability,
        parameters,
        initial_positions,
        num_steps=num_samples,
    )

    for par in ["gp_parameter", "lc_parameter"]:
        chain = np.array(state_history.position[par])
        chains[par].append(chain)

np.savez(f"saved/{model_name}_parameter.npz", **chains)